# 02 — Visualizando Mapas LCZ e Quantificando Áreas por Classe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/02_visualization_area_stats.pt.ipynb)

**Objetivo de aprendizagem**: aprender a transformar um raster bruto de classificação LCZ em duas entregas que planejadores urbanos e revisores realmente precisam — um mapa interativo e navegável que qualquer pessoa pode explorar no navegador, e uma quantificação de quanta área cada classe LCZ ocupa. Ao final deste notebook você será capaz de renderizar um mapa LCZ com `lcz_plot_map`, alternar entre os modos de visualização de classes discretas e raster contínuo, e calcular/plotar áreas por classe com `lcz_cal_area` usando cinco estilos de gráfico diferentes.

## Por que a visualização interativa importa para a comunicação científica de LCZ

O esquema de Zonas Climáticas Locais (Stewart & Oke, 2012) foi projetado como uma classificação *padronizada* e comparável mundialmente de paisagens urbanas e naturais em 17 classes (1–10 tipos construídos, de alta densidade compacta a baixa densidade extensa; A–G / 11–17 coberturas naturais, de árvores densas a corpos d'água). Uma classificação só é útil, no entanto, se as pessoas que precisam agir sobre ela — planejadores urbanos, prefeituras, gestores de saúde pública, equipes de adaptação climática — puderem de fato lê-la. Um PNG estático enterrado em um relatório em PDF raramente sobrevive à viagem da mesa do pesquisador até a reunião de planejamento; um mapa interativo que um interessado pode arrastar, ampliar e passar o mouse para ver a classe exata em uma esquina específica, sobrevive.

`lcz_plot_map` (LCZ4py, seguindo a filosofia de visualização do pacote R LCZ4r, Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) renderiza rasters LCZ como figuras Plotly WebGL — HTML autocontido que abre em qualquer navegador, com tooltips ao passar o mouse, zoom/pan, e (via a opção `renderer='maplibre'`) pode sobrepor a classificação em um mapa base real do OpenStreetMap para contexto geográfico imediato.

**Raster discreto vs. contínuo.** Dois tipos fundamentalmente diferentes de raster aparecem em fluxos de trabalho LCZ, e `lcz_plot_map` trata ambos:

- **Mapa coroplético de classes discretas**: o próprio mapa LCZ — cada pixel contém um código de classe inteiro (1–17). A forma correta de renderizar isso é um mapa de cores categórico, com uma cor sólida por classe e uma legenda listando os nomes das classes, *não* um gradiente contínuo de cores (um gradiente implicaria visualmente que a classe 5 está 'entre' as classes 4 e 6, o que não faz sentido — os códigos LCZ são uma escala nominal, não ordinal).
- **Sobreposição de raster contínuo**: campos físicos derivados na mesma grade, por exemplo a Temperatura de Superfície (LST) por pixel de `lcz_get_lst`, ou um parâmetro morfológico de `lcz_get_parameters`. Esses são dados ordinais/intervalares, então uma escala de cores divergente ou sequencial (ex. `RdBu_r`, azul=frio a vermelho=quente) com barra de cores é a renderização correta, e calcular médias ou interpolar entre valores de pixels faz sentido.

O parâmetro `data_type` de `lcz_plot_map` (`'auto'`, `'lcz'`, `'continuous'`) escolhe o modo correto automaticamente, inspecionando o tipo de dado do raster, mas você pode forçá-lo explicitamente. Confundir os dois — por exemplo, aplicar uma paleta categórica em um raster de temperatura — é uma fonte comum e evitável de figuras enganosas em climatologia urbana.

**Por que a quantificação de área vem logo após a visualização.** Um mapa responde *onde*; um gráfico de barras/pizza/rosca/sunburst/treemap de área por classe responde *quanto*. Antes de qualquer análise mais profunda de clima urbano (intensidade de UHI, modelagem de conforto térmico, extração de parâmetros morfológicos), é prática padrão perguntar primeiro: qual fração desta cidade é alta densidade compacta (LCZ 1) vs. baixa densidade aberta (LCZ 6) vs. árvores densas (LCZ A/11) vs. água (LCZ G/17)? `lcz_cal_area` calcula a área exata por classe em km² (levando em conta o tamanho real dos pixels e o CRS do raster) e a renderiza em um de cinco tipos de gráfico interativos, cada um adequado a uma necessidade de comunicação diferente.

## Instalar o LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Obter um mapa LCZ real para trabalhar

Usamos Juiz de Fora (Liechtenstein) — uma cidade pequena cuja caixa delimitadora mantém o download rápido — para que este notebook execute de ponta a ponta em poucos minutos. Tudo abaixo se aplica sem alterações a qualquer cidade.

In [2]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Juiz de Fora")
print(map_path)

13:34:18 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_c7031cb44f08aaf6.tif


## `lcz_plot_map` — mapa interativo de classes discretas

Parâmetros principais:
- `x`: caminho para o GeoTIFF LCZ (ou um dataset rasterio já aberto).
- `show_legend`: mostra a legenda das 17 classes (recomendado, ligado por padrão).
- `inclusive`: troca a paleta LCZ padrão por uma paleta acessível a daltônicos (veja abaixo).
- `use_webgl` / `max_pixels`: mantêm rasters grandes responsivos, renderizando via WebGL e reduzindo a resolução acima de um limite de pixels.
- `renderer`: `'plotly'` (figura autocontida, padrão) ou `'maplibre'` (página HTML com mapa base OpenStreetMap por baixo — bom para dar referência geográfica a interessados).
- `data_type`: `'auto'` (padrão) detecta LCZ vs. dado contínuo pelo tipo de dado do raster.

Retorna um `LCZPlotResult` com `.fig` (figura Plotly) e, para `maplibre`, `.html`. `result.show()` exibe qualquer um dos dois.

In [3]:
from LCZ4py import lcz_plot_map

result = lcz_plot_map(map_path, show_legend=True, title="Mapa LCZ — Juiz de Fora")
result.fig.show()

**Lendo este mapa**: cada mancha de cor sólida é uma classe LCZ — vermelhos/laranjas quentes para os tipos construídos (baixa densidade compacta ou aberta são típicos de uma pequena cidade alpina como Juiz de Fora), verdes para classes vegetadas (árvores densas/esparsas nas encostas ao redor), e azul para água/rio. Passe o mouse sobre qualquer pixel para ver sua classe exata na tooltip. Para uma cidade deste porte, espera-se que a pegada construída seja uma fração pequena da área total mapeada, com classes naturais (floresta, plantas baixas, rocha exposta) dominando — um padrão bem diferente de um mapa LCZ de metrópole densa.

## A paleta acessível a daltônicos (`inclusive=True`)

A paleta padrão do esquema LCZ (Stewart & Oke 2012) usa vermelhos/laranjas para classes construídas e verdes/azuis para naturais — um esquema difícil ou impossível de distinguir para os ~4-8% dos leitores com deficiência de visão de cores vermelho-verde (a forma mais comum de daltonismo). Definir `inclusive=True` troca para uma paleta perceptualmente distinta e segura para daltônicos, preservando os mesmos limites de classe e a legenda. Isso custa nada no código (uma única palavra-chave) e importa muito para a acessibilidade de figuras publicadas, pôsteres de conferência e qualquer material compartilhado com um público geral — a comunicação científica acessível deveria ser o padrão, não uma exceção reservada para quem pede explicitamente.

In [4]:
result_inclusive = lcz_plot_map(map_path, show_legend=True, inclusive=True,
                                title="Mapa LCZ — Juiz de Fora (paleta acessível a daltônicos)")
result_inclusive.fig.show()

Compare as duas figuras acima: os limites de classe são idênticos, apenas a codificação de cores mudou. Em qualquer relatório ou artigo destinado a um público amplo, prefira `inclusive=True`.

## Estatísticas de área por classe com `lcz_cal_area`

Parâmetros principais:
- `x`: caminho para o GeoTIFF LCZ.
- `plot_type`: `'bar'`, `'pie'`, `'donut'`, `'sunburst'`, ou `'treemap'`.
- `iplot`: se `False`, pula a renderização do gráfico e retorna apenas o `DataFrame` de área.
- `inclusive`: paleta acessível a daltônicos, igual a `lcz_plot_map`.
- `use_duckdb` / `use_geoarrow`: backends acelerados opcionais para rasters grandes (degradam graciosamente para apenas Polars se não instalados).

Retorna um `LCZAreaResult` com `.df` (um DataFrame Polars: classe LCZ, nome, área em km², porcentagem da área mapeada), `.fig` (o gráfico Plotly), e `.geoarrow_table` (exportação opcional de polígonos com cópia zero). Este é o primeiro passo quantitativo padrão em qualquer estudo de clima urbano baseado em LCZ: antes de calcular a intensidade de UHI ou parâmetros morfológicos, você quer saber qual fração da cidade é cobertura urbana vs. natural.

### Gráfico de barras — melhor para comparação precisa classe a classe

Um gráfico de barras é o padrão certo sempre que o público precisa comparar magnitudes exatas entre todas as 17 classes, ou ranquear classes por área. Barras tornam fáceis de ler pequenas diferenças entre classes adjacentes, o que gráficos de pizza não conseguem.

In [5]:
from LCZ4py import lcz_cal_area

area_bar = lcz_cal_area(map_path, plot_type="bar", title="Área por classe LCZ — Juiz de Fora")
area_bar.fig.show()
area_bar.df

lcz,count,area_km2,area_perc,lcz_name,lcz_col,lcz_colorblind
i64,i64,f64,f64,str,str,str
2,42,0.39,0.03,"""Compact midrise""","""#D9081C""","""#D8755E"""
3,2852,26.51,1.83,"""Compact lowrise""","""#FF0A22""","""#C98027"""
4,4,0.04,0.0,"""Open highrise""","""#C54F1E""","""#B48C00"""
5,40,0.37,0.03,"""Open midrise""","""#FF6628""","""#989600"""
6,6078,56.51,3.91,"""Open lowrise""","""#FF985E""","""#739F00"""
…,…,…,…,…,…,…
13,9,0.08,0.01,"""Bush, scrub""","""#628432""","""#6290E5"""
14,10066,93.58,6.48,"""Low plants""","""#B5DA7F""","""#9E7FE5"""
15,50,0.46,0.03,"""Bare rock or paved""","""#000000""","""#C36FDA"""


**Lendo isto**: as barras mais altas identificam a(s) cobertura(s) de solo dominante(s) na área de estudo. Para Juiz de Fora, espera-se que uma ou duas classes naturais (floresta/árvores densas, plantas baixas) dominem, com cada classe construída contribuindo com uma fração comparativamente pequena — confirmação quantitativa do que o mapa mostrou visualmente acima.

### Gráfico de rosca — melhor para uma visão rápida de 'urbano vs. natural'

Um gráfico de rosca (ou pizza) é preferível quando a mensagem é uma história de *proporção* — por exemplo, "que fração da cidade é construída?" — em vez de um ranqueamento preciso. O enquadramento visual de todo-para-parte comunica isso imediatamente a um público não técnico (câmara municipal, comunicado de imprensa), ao custo de tornar comparações entre classes pequenas mais difíceis do que um gráfico de barras.

In [6]:
area_donut = lcz_cal_area(map_path, plot_type="donut", inclusive=True,
                          title="Participação de área por classe LCZ — Juiz de Fora")
area_donut.fig.show()

**Lendo isto**: o tamanho angular de cada fatia é proporcional à sua participação na área. Agrupar mentalmente as fatias em 'cores quentes' (construído, LCZ 1-10) vs. 'cores frias' (natural, LCZ A-G/11-17) dá uma leitura instantânea do equilíbrio urbano-vs-natural — exatamente o número que um planejador pede primeiro.

### Treemap — melhor para mostrar hierarquia e área relativa com detalhe aninhado

Um treemap codifica a participação de área como tamanho de retângulo (como uma rosca) mas também suporta naturalmente agrupar classes em uma hierarquia pai/filho (ex. "Construído" e "Natural" como grupos pais, classes LCZ individuais aninhadas dentro). É a escolha preferível quando você quer tanto a história todo-para-parte *quanto* um layout compacto e denso em informação que também mostra de forma legível o rótulo e valor exatos de cada classe na mesma figura — útil em um relatório escrito onde uma página inteira está disponível mas você não quer uma parede de barras.

In [7]:
area_treemap = lcz_cal_area(map_path, plot_type="treemap",
                            title="Área por classe LCZ — Juiz de Fora (treemap)")
area_treemap.fig.show()

**Lendo isto**: a área do retângulo é proporcional à área mapeada de cada classe; retângulos maiores são as classes dominantes. Diferente do gráfico de barras, classes pequenas continuam visíveis (como pequenos blocos) em vez de comprimidas contra um eixo, o que é útil quando você precisa que toda classe seja rotulada em uma única exportação estática para uma figura de relatório.

### Obtendo apenas os números (sem gráfico)

Defina `iplot=False` quando você só precisa da tabela de área — por exemplo, para alimentar uma análise estatística posterior ou uma figura customizada em outro ponto do pipeline.

In [8]:
area_df = lcz_cal_area(map_path, iplot=False)
area_df

lcz,count,area_km2,area_perc,lcz_name,lcz_col,lcz_colorblind
i64,i64,f64,f64,str,str,str
2,42,0.39,0.03,"""Compact midrise""","""#D9081C""","""#D8755E"""
3,2852,26.51,1.83,"""Compact lowrise""","""#FF0A22""","""#C98027"""
4,4,0.04,0.0,"""Open highrise""","""#C54F1E""","""#B48C00"""
5,40,0.37,0.03,"""Open midrise""","""#FF6628""","""#989600"""
6,6078,56.51,3.91,"""Open lowrise""","""#FF985E""","""#739F00"""
…,…,…,…,…,…,…
13,9,0.08,0.01,"""Bush, scrub""","""#628432""","""#6290E5"""
14,10066,93.58,6.48,"""Low plants""","""#B5DA7F""","""#9E7FE5"""
15,50,0.46,0.03,"""Bare rock or paved""","""#000000""","""#C36FDA"""


## Conclusão

Você agora tem duas ferramentas complementares para comunicar uma classificação LCZ: `lcz_plot_map` para uma visão espacial interativa e explorável (com uma opção acessível a daltônicos via `inclusive=True`, e tanto um modo de classes discretas para mapas LCZ quanto um modo contínuo para rasters como LST), e `lcz_cal_area` para quantificação precisa de área por classe em cinco estilos de gráfico (barras para comparação precisa, rosca/pizza para histórias rápidas de proporção, sunburst/treemap para layouts hierárquicos ou densos em informação). Juntas, formam a primeira entrega padrão de qualquer estudo de clima urbano baseado em LCZ: *onde* estão as classes, e *quanto* de cada uma cobre a cidade.

**Anterior**: `01_map_acquisition` (download e obtenção de mapas LCZ).
**Próximo**: `03_morphological_parameters` — extraindo os 34 parâmetros morfológicos/térmicos (fração de área construída, fator de visão do céu, rugosidade, etc.) que fundamentam cada classe LCZ.